## Continual Learning via On-Policy Self-Distillation

A reproduction of the "Distillation for Personalization" experiment from
**Kevin Lu et al., "On-Policy Distillation" (Thinking Machines, Oct 2025)**,
adapted to small-scale hardware (GTX 1650, 4 GB VRAM).

> baseline eval → domain SFT → catastrophic forgetting → on-policy distillation recovery

1. **Baseline** — measure an instruction-tuned model's behavior on two axes:
   - Instruction-following (IF): can it obey diverse user constraints?
   - JSON: does it emit structured JSON when asked?
2. **Domain SFT** — fine-tune the same model on synthetic JSON data.
   *Hypothesis:* JSON score rises, IF score collapses (catastrophic forgetting).
3. **On-policy distillation recovery** — distill the SFT'd student against
   the **original, pre-SFT version of itself** as teacher, on general
   instruction prompts. *Hypothesis:* IF score recovers, JSON score preserved.

The "earlier self as teacher" trick is the original insight from the TM blog.
We never use a larger or smarter teacher — we use the same-architecture
checkpoint *before* domain fine-tuning. This makes the experiment cheap and
demonstrates that on-policy distillation is a behavior-recovery tool, not
just a compression tool.

### Models
- Student (trainable): `Qwen/Qwen2.5-0.5B-Instruct` in FP16
- Teacher (frozen): same model, original weights, in 4-bit NF4

### Method
- SFT phase: standard next-token cross-entropy, prompt tokens masked from loss
- OPD phase: per-token reverse KL between student and teacher distributions,
  computed only on student-rolled-out tokens (the on-policy part)

In [1]:
# 1: Setup Environment
import os, re, json, random
import torch
import torch.nn.functional as F
import numpy as np
import bitsandbytes as bnb
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"• PyTorch: {torch.__version__}")
print(f"• GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

• PyTorch: 2.9.0+cu130
• GPU: NVIDIA GeForce GTX 1650


#### 2: Load the _student_ model:
`Qwen/Qwen2.5-0.5B-Instruct` its weights will move through three states: baseline → SFT'd on JSON → distillation-recovered

In [2]:
STUDENT_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"

student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"pad: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"eos: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")

assert tokenizer.pad_token_id != tokenizer.eos_token_id, "pad and eos must differ"
print("✓ Student loaded")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+cu130).
W0520 17:00:43.767000 13296 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


pad: '<|endoftext|>' (id=151643)
eos: '<|im_end|>' (id=151645)
✓ Student loaded


#### 3: Build evaluators (Phase 1)

1. Measure `IF score` and `JSON score` across all three stages:
   - Baseline (pre-SFT): measure IF + JSON scores
   - Domain SFT: measure IF + JSON scores (post-SFT)
   - On-policy distillation: measure IF + JSON scores (post-OPD)


In [3]:
# Instruction-Following Evaluator
IF_TESTS = [
    {"prompt": "Respond using exactly five words. No more, no less.",
     "check": lambda r: len(r.strip().split()) == 5},
    {"prompt": "Include the word 'banana' somewhere in your response.",
     "check": lambda r: "banana" in r.lower()},
    {"prompt": "Respond entirely in uppercase letters.",
     "check": lambda r: any(c.isalpha() for c in r) and r.strip() == r.strip().upper()},
    {"prompt": "Answer with just 'yes' or 'no', nothing else.",
     "check": lambda r: r.strip().lower().rstrip('.!') in {"yes", "no"}},
    {"prompt": "Tell me a fact about the moon. End your response with the word 'fascinating'.",
     "check": lambda r: r.strip().lower().rstrip('.!?').endswith("fascinating")},
    {"prompt": "List three colors, one per line, with no other text.",
     "check": lambda r: len([l for l in r.strip().split('\n') if l.strip()]) == 3},
    {"prompt": "Respond with exactly two sentences.",
     "check": lambda r: len([s for s in re.split(r'[.!?]+', r) if s.strip()]) == 2},
    {"prompt": "Repeat the word 'echo' exactly four times, separated by spaces.",
     "check": lambda r: r.strip().lower() == "echo echo echo echo"},
    {"prompt": "What is 2+2? Answer with just the number.",
     "check": lambda r: r.strip().rstrip('.') == "4"},
    {"prompt": "Write a sentence where every word starts with the letter S.",
     "check": lambda r: len(r.strip().split()) >= 3 and all(w[0].lower() == 's' for w in re.findall(r"[A-Za-z]+", r))},
    {"prompt": "Name one country in Africa. Just the country name, nothing else.",
     "check": lambda r: r.strip().rstrip('.').lower() in {
         "egypt","nigeria","kenya","morocco","ethiopia","ghana","tanzania","uganda",
         "south africa","algeria","tunisia","senegal","zambia","zimbabwe","mali",
         "sudan","cameroon","angola","mozambique","botswana","rwanda","libya"}},
    {"prompt": "Reply with exactly the phrase: I AM A ROBOT",
     "check": lambda r: r.strip().rstrip('.!') == "I AM A ROBOT"},
]

# JSON Evaluator

JSON_TESTS = [
    "Create a JSON object for a person named Alice Johnson, age 35, who lives in Seattle and works as an engineer.",
    "Output JSON for a book titled '1984' by George Orwell, published in 1949, genre dystopian.",
    "Convert this product info to JSON: a Sony camera, costs $899.99, in stock.",
    "Make a JSON object: name=Bob, age=42, city=Tokyo, occupation=chef.",
    "Output as JSON: title 'Dune', author Frank Herbert, year 1965.",
    "Create JSON for a laptop, brand Lenovo, price $1200, available.",
    "JSON format: person named Maya, 28 years old, lives in London, works as a designer.",
    "Output JSON: product name 'headphones', brand Bose, price 350 dollars, in stock true.",
    "Make a JSON object describing the book 'The Hobbit' by J.R.R. Tolkien, published 1937.",
    "JSON for: Carlos Rodriguez, 50, lives in Berlin, lawyer.",
]

def extract_json(text):
    """Return a dict if text contains a parseable JSON object, else none"""
    text = text.strip()
    # Strip markdown fences if present
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*\n?", "", text)
        text = re.sub(r"\n?```\s*$", "", text)
        text = text.strip()
    # Try parsing whole text
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and len(obj) > 0: return obj
    except json.JSONDecodeError:
        pass
    # Try first {...} block
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            if isinstance(obj, dict) and len(obj) > 0: return obj
        except json.JSONDecodeError:
            pass
    return None

# Generic eval runner

def _generate(model, prompt_text, max_new_tokens=80):
    inp = tokenizer([prompt_text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

def evaluate_if(model, verbose=False):
    model.eval()
    passed = 0
    for t in IF_TESTS:
        msgs = [{"role": "user", "content": t["prompt"]}]
        formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        resp = _generate(model, formatted, max_new_tokens=60)
        ok = t["check"](resp)
        if ok: passed += 1
        if verbose:
            mark = "✓" if ok else "✗"
            print(f"  {mark}  {resp[:70]!r}")
    return passed / len(IF_TESTS) * 100

def evaluate_json(model, verbose=False):
    model.eval()
    passed = 0
    for prompt in JSON_TESTS:
        msgs = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        resp = _generate(model, formatted, max_new_tokens=120)
        ok = extract_json(resp) is not None
        if ok: passed += 1
        if verbose:
            mark = "✓" if ok else "✗"
            print(f"  {mark}  {resp[:80]!r}")
    return passed / len(JSON_TESTS) * 100

print("✓ Evaluators built")


✓ Evaluators built


In [4]:
# 4: Baseline Evaluation
print("BASELINE (no training yet)")
print("-" * 40)

print("\n[IF eval]")
baseline_if = evaluate_if(student_model, verbose=True)
print(f"\n  → IF score: {baseline_if:.1f}%")

print("\n[JSON eval]")
baseline_json = evaluate_json(student_model, verbose=True)
print(f"\n -> JSON score: {baseline_json:.1f}%")

scores = {"baseline": {"if": baseline_if, "json": baseline_json}}

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASELINE (no training yet)
----------------------------------------

[IF eval]
  ✗  'Understood.'
  ✓  'Certainly! Here\'s an example of how you might incorporate "banana" int'
  ✓  'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
  ✓  'Yes'
  ✗  'The moon is fascinating for its unique position in our solar system an'
  ✓  'Red\nBlue\nYellow'
  ✓  'I am Qwen, an artificial intelligence designed to assist and provide i'
  ✓  'echo echo echo echo'
  ✓  '4'
  ✓  'Surely, swiftly, steadily, silently, successfully.'
  ✓  'Egypt'
  ✓  'I AM A ROBOT'

  → IF score: 83.3%

[JSON eval]
  ✗  'Here is the JSON object for Alice Johnson:\n```json\n{\n  "name": "Alice Johnson",\n'
  ✓  '```json\n{\n  "title": "1984",\n  "author": "George Orwell",\n  "year": 1949,\n  "gen'
  ✓  '```json\n{\n  "product": {\n    "name": "Sony Camera",\n    "price": "$899.99",\n    '
  ✓  'Here is the JSON object you requested:\n\n```json\n{\n  "name": "Bob",\n  "age": 42,\n'
  ✓  '```json\n{\n  "title": "Dune",\n  "author": "Frank Herbert

#### 5. Domain SFT on JSON (Phase 2)
SFT on synthetic (prompt, json_response) pairs. The data is generated programmatically — no teacher needed for this phase because the labels are perfect by construction.

Loss is masked: we only train on the response tokens, not the prompt. This is the same masking trick as in TRL's `SFTTrainer`.

In [8]:
FIRST = ["Alice","Bob","Carlos","Diana","Ethan","Fatima","Grace","Hiroshi",
        "Isabella","Jamal","Kira","Liam","Maya","Noah","Olivia","Priya","Raj","Sofia"]
LAST = ["Johnson","Smith","Rodriguez","Patel","Chen","Williams","Brown","Garcia",
        "Martinez","Miller","Davis","Wilson","Kumar","Singh","Lee"]
CITIES = ["New York","London","Tokyo","Paris","Mumbai","Sydney","Berlin","Toronto",
          "Seattle","Boston","Madrid","Singapore","Dubai","Amsterdam"]
JOBS = ["software engineer","doctor","teacher","artist","chef","writer","designer",
        "lawyer","architect","biologist","photographer","accountant","pilot"]
PRODUCTS = ["laptop","phone","book","headphones","watch","camera","tablet","keyboard","monitor"]
BRANDS = ["Sony","Apple","Samsung","Bose","Canon","Lenovo","Dell","HP","Logitech"]
BOOKS = [("Frankenstein","Mary Shelley",1818),("Dune","Frank Herbert",1965),
         ("The Hobbit","J.R.R. Tolkien",1937),("1984","George Orwell",1949),
         ("Brave New World","Aldous Huxley",1932),("Foundation","Isaac Asimov",1951),
         ("Beloved","Toni Morrison",1987),("Things Fall Apart","Chinua Achebe",1958)]
GENRES = ["fiction","fantasy","scifi","drama","mystery","historical","biography"]

def make_person():
    p = {"name": f"{random.choice(FIRST)} {random.choice(LAST)}",
         "age": random.randint(20, 70),
         "city": random.choice(CITIES),
         "occupation": random.choice(JOBS)}
    prompt = (f"Create a JSON object for a person named {p['name']}, "
              f"age {p['age']}, living in {p['city']}, working as a {p['occupation']}.")
    return prompt, json.dumps(p)

def make_product():
    p = {"name": random.choice(PRODUCTS),
         "brand": random.choice(BRANDS),
         "price": round(random.uniform(50, 2000), 2),
         "in_stock": random.choice([True, False])}
    prompt = (f"Convert this product info to JSON: {p['brand']} {p['name']}, "
              f"costs ${p['price']}, {'in stock' if p['in_stock'] else 'out of stock'}.")
    return prompt, json.dumps(p)

def make_book():
    title, author, year = random.choice(BOOKS)
    b = {"title": title, "author": author, "year": year, "genre": random.choice(GENRES)}
    prompt = f"Output JSON for the book '{title}' by {author}, published {year}, genre {b['genre']}."
    return prompt, json.dumps(b)

GENS = [make_person, make_product, make_book]

def make_sft_data(n):
    return [{"prompt": (g := random.choice(GENS))()[0] if False else None,
             "response": None} for _ in range(0)] or \
           [(lambda pair: {"prompt": pair[0], "response": pair[1]})(random.choice(GENS)()) for _ in range(n)]

# def make_sft_data(n):
#     out = []
#     for _ in range(n):
#         prompt, response = random.choice(GENS)()
#         out.append({"prompt": prompt, "response": response})
#     return out

# Generate 800 examples
sft_data = make_sft_data(200)
print(f"Generated {len(sft_data)} SFT examples")
print("\nSample:")
print(f"  Prompt:   {sft_data[0]['prompt']}")
print(f"  Response: {sft_data[0]['response']}")

Generated 200 SFT examples

Sample:
  Prompt:   Output JSON for the book '1984' by George Orwell, published 1949, genre fiction.
  Response: {"title": "1984", "author": "George Orwell", "year": 1949, "genre": "fiction"}


In [9]:
# 6:  SFT dataset, collator, and training loop
SFT_SYSTEM = "You are a helpful assistant that outputs valid JSON objects."

class SFTDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.items = []
        for ex in examples:
            full_msgs = [
                {"role": "system", "content": SFT_SYSTEM},
                {"role": "user", "content": ex["prompt"]},
                {"role": "assistant", "content": ex["response"]},
            ]
            prompt_msgs = full_msgs[:-1]  # everything except the assistant turn
            full_text = tokenizer.apply_chat_template(full_msgs, tokenize=False, add_generation_prompt=False)
            prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
            self.items.append((full_text, prompt_text))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def sft_collate(batch, tokenizer, max_len=256):
    """left-pad, build labels with -100 on prompt_padding positions"""
    full_tok = [tokenizer(f, truncation=True, max_length=max_len)["input_ids"] for f, _ in batch ]
    prompt_tok = [tokenizer(p, truncation=True, max_length=max_len)["input_ids"] for _, p in batch]
    max_L = max(len(t) for t in full_tok)
    pad_id = tokenizer.pad_token_id

    input_ids, attn, labels = [], [], []

    for full, prompt in zip(full_tok, prompt_tok):
        L, P = len(full), len(prompt)
        pad_n = max_L - L
        # left-pad
        input_ids.append([pad_id] * pad_n + full)
        attn.append([0]*pad_n + [1]*L)
        # labels: -100 on left pad and prompt position; real ids on response
        labels.append([-100]*(pad_n + P) + full[P:])

    return{
        "inputs_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

sft_dataset = SFTDataset(sft_data, tokenizer)
sft_loader = DataLoader(sft_dataset, batch_size=2, shuffle=True, collate_fn=lambda b: sft_collate(b, tokenizer) )

sft_optim = bnb.optim.PagedAdamW8bit(student_model.parameters(), lr=2e-5)

print("Starting SFT phase...")
student_model.train()
sft_losses = []

for epoch in range(1):
    for step, batch, in enumerate(sft_loader):
        sft_optim.zero_grad()
        ids = batch["inputs_ids"].to("cuda")
        mask = batch["attention_mask"].to("cuda")
        lbls = batch["labels"].to("cuda")

        logits = student_model(input_ids = ids, attention_mask = mask).logits

        # Shift for next token prediction
        shift_logits = logits[:, :-1, :].contiguous().float()
        shift_labels = lbls[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index= -100,
        )
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"UNSTABLE STEP {step}, SKIPPED")
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
        sft_optim.step()
        sft_losses.append(loss.item())
        
        if step % 20 == 0:
            print(f"    SFT step {step:4d} | loss: {loss.item():.4f}")

print(f"\n✓ SFT done. Final avg (last 20): {np.mean(sft_losses[-20:]):.4f} ")

Starting SFT phase...
    SFT step    0 | loss: 0.0000
    SFT step   20 | loss: 0.0000
    SFT step   40 | loss: 0.0000
    SFT step   60 | loss: 0.0000
    SFT step   80 | loss: 0.0000

✓ SFT done. Final avg (last 20): 0.0000 


In [10]:
# 7: Post-SFT evaluation
print("POST-SFT (after JSON domain training)")
print("-" * 40)

print("\n[IF eval] (expect: drops sharply)")
post_sft_if = evaluate_if(student_model, verbose=True)
print(f"\n  → IF: {post_sft_if:.1f}%   (was {baseline_if:.1f}%)")

print("\n[JSON eval] (expect: rises toward 100%)")
post_sft_json = evaluate_json(student_model, verbose=True)
print(f"\n  → JSON: {post_sft_json:.1f}%   (was {baseline_json:.1f}%)")

scores["post_sft"] = {"if": post_sft_if, "json": post_sft_json}

POST-SFT (after JSON domain training)
----------------------------------------

[IF eval] (expect: drops sharply)
  ✗  '{"name": "{"name": "{"name": "{"name": "{"name": "{"name": "{"name": "'
  ✓  '{"name": "banana", "brand": "Alibaba", "price": 100, "in_stock": true}'
  ✗  '{"name": "{"name": "{"name": "{"name": "{"name": "{"name": "{"name": "'
  ✗  '{"name": "yes", "age": 10, "{"age": 10}'
  ✗  '{"name": "moon", "age": 10, "description": "{"name": "moon", "age": 10'
  ✗  '{"name": "color", "brand": "Alibaba Cloud", "price": 1, "in_stock": tr'
  ✗  '{"name": "{"name": "{"name": "{"name": "{"name": "{"name": "{"name": "'
  ✗  '{"name": "echo", "repeat": true}'
  ✗  '{"name": "2+", "age": true}'
  ✗  '{"name": "S", "age": 10, "sentence": "{"title": "S", "author": "{"name'
  ✗  '{"name": "country", "{"name": "country", "{"name": "country", "{"name"'
  ✗  '{"name": "ROBOT", "age": 100, "city": "I AM A ROBOT"}'

  → IF: 8.3%   (was 83.3%)

[JSON eval] (expect: rises toward 100%)
  ✓  '{"na

#### 8: On-policy distillation recovery (Phase 3)
Thinking Machine blog's central Ploy: we load a **fresh copy of the original** `Qwen2.5-0.5B-Instruct` (the same architecture, the same parameter count, but the pre-SFT weights) as the teacher. The student is the SFT'd model from Phase 2.

We then run on-policy distillation on general instruction-following prompts
(diverse, NOT JSON-related). The student rolls out trajectories under its
current degraded behavior; the teacher scores each token with the policy it
still has from its original instruction tuning. The reverse-KL loss pulls
the student's distribution back toward the teacher's — recovering the lost
instruction-following capability.

The teacher is loaded in 4-bit to save VRAM (we now hold two models on the GPU).
The student stays in FP16 because it's the one receiving gradients.

In [11]:
# Load original model as teacher
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

teacher_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID, # original loaded model from HF
    quantization_config = bnb_config,
    device_map = "auto",
    trust_remote_code = True,
)

teacher_model.train()

print("✓ Original Qwen2.5-0.5B-Instruct loaded as TEACHER (4-bit, frozen)")
print(f"  Student device: {next(student_model.parameters()).device}")
print(f"  Teacher device: {next(teacher_model.parameters()).device}")

✓ Original Qwen2.5-0.5B-Instruct loaded as TEACHER (4-bit, frozen)
  Student device: cuda:0
  Teacher device: cuda:0


In [12]:
# 9: Recovery prompts
RECOVERY_PROMPTS = [
    "Explain what gravity is in one short paragraph.",
    "Write a short poem about autumn.",
    "What are three benefits of regular exercise?",
    "Give me a simple recipe for scrambled eggs.",
    "Summarize the plot of Romeo and Juliet in two sentences.",
    "What is the capital of France?",
    "Name three famous painters from the Renaissance.",
    "How does photosynthesis work? Keep it short.",
    "Recommend a book for someone who likes adventure novels.",
    "What's the difference between a virus and bacteria?",
    "Tell me a joke about computers.",
    "Write a one-sentence description of the Pacific Ocean.",
    "List five common programming languages.",
    "What year did World War II end?",
    "Describe the taste of coffee in one sentence.",
    "Why is the sky blue? Brief answer please.",
    "What is the largest planet in our solar system?",
    "Suggest a healthy breakfast option.",
    "How many continents are there? Name them.",
    "Explain what a metaphor is with one example.",
    "Write a short congratulatory message for a graduation.",
    "What does 'serendipity' mean?",
    "Name three benefits of reading books.",
    "Compose a short greeting for a birthday card.",
    "What is artificial intelligence? Keep it simple.",
    "Suggest a good movie for a rainy weekend.",
    "How do plants reproduce? Quick summary.",
    "Tell me an interesting fact about octopuses.",
    "Write a polite request to reschedule a meeting.",
    "What's a quick way to memorize new vocabulary?",
    "Describe a sunset in two sentences.",
    "What is the speed of light?",
    "List three traditional dishes from Italy.",
    "Why do we dream? Brief answer.",
    "Name three Nobel Prize winners in literature.",
    "What is mindfulness?",
    "Recommend a hobby for someone who likes being outdoors.",
    "Explain the water cycle in three sentences.",
    "What does the word 'ephemeral' mean?",
    "Write a brief apology for being late.",
    "Tell me a tongue twister.",
    "What's the difference between weather and climate?",
    "Suggest a way to stay focused while studying.",
    "Name four wonders of the ancient world.",
    "What is democracy?",
    "Write a haiku about the ocean.",
    "Describe the smell of fresh bread.",
    "What is the boiling point of water in Celsius?",
    "Name three primary colors.",
    "How do bees make honey? Brief explanation.",
    "Suggest a thoughtful gift for a teacher.",
    "What is the meaning of 'carpe diem'?",
    "Write a short motivational quote.",
    "Name three popular tourist destinations in Asia.",
    "What is the longest river in the world?",
    "Explain how a battery works in simple terms.",
    "Suggest a quick exercise routine for the morning.",
    "What's a good way to learn a musical instrument?",
    "Describe the feeling of rain in summer.",
    "Name three classic novels everyone should read.",
    "What is the capital of Australia?",
    "How does a refrigerator work? Brief explanation.",
    "Tell me an interesting fact about space.",
    "Write a short thank-you note for a gift.",
    "What is photosynthesis in one sentence?",
    "Suggest a podcast topic worth exploring.",
    "Name three types of clouds.",
    "What is the Pythagorean theorem?",
    "Describe what makes a good leader in two sentences.",
    "How can someone improve their writing skills?",
    "Tell me a fact about the human brain.",
    "What is renewable energy?",
    "Suggest a way to be more productive.",
    "Name three famous scientists.",
    "What is the function of the heart?",
    "Write a short pep talk for a nervous student.",
    "Describe the color green using a metaphor.",
    "Name three musical instruments.",
    "How long does it take to travel to the moon?",
    "What is the role of a vaccine?",
    "Suggest a quick lunch idea for a busy day.",
]

In [ ]:
# 10: OPD training loop
class RecoveryDataset(Dataset):
    def __init__(self, prompts, tokenizer):
        sys_msg = {"role": "system", "content": "You are a helpful assistant."}
        self.formatted = [
            tokenizer.apply_chat_template(
                [sys_msg, {"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True
            )
            for p in prompts
        ]
    def __len__(self): return len(self.formatted)
    def __getitem__(self, i): return self.formatted[i]

recovery_ds = RecoveryDataset(RECOVERY_PROMPTS, tokenizer)
recovery_loader = DataLoader(recovery_ds, batch_size=1, shuffle=True)

opd_optim = bnb.optim.PagedAdamW8bit(student_model.parameters(), lr=5e-6)

print(f"Starting OPD recovery: {len(RECOVERY_PROMPTS)} prompts × 2 epochs = {len(RECOVERY_PROMPTS)*2} steps")
student_model.train()
opd_losses = []

for epoch in range(2):
    for step, batch_prompts in enumerate(recovery_loader):
        opd_optim.zero_grad()

        # ─── Phase 1: student rollout ───
        s_inp = tokenizer(list(batch_prompts), return_tensors="pt", padding=True)
        s_ids = s_inp["input_ids"].to("cuda")
        s_mask = s_inp["attention_mask"].to("cuda")
        s_plen = s_ids.shape[1]

        with torch.no_grad():
            rollout = student_model.generate(
                input_ids=s_ids, attention_mask=s_mask,
                max_new_tokens=64,
                do_sample=True, temperature=0.7,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        traj = rollout
        suffix = traj[:, s_plen:]
        traj_mask = traj.ne(tokenizer.pad_token_id).long()

        # ─── Phase 2: forward both models on the SAME trajectory ───
        # (Same tokenizer, same prompt — no splice trick needed here)
        s_logits = student_model(traj, attention_mask=traj_mask).logits
        with torch.no_grad():
            t_logits = teacher_model(traj, attention_mask=traj_mask).logits

        # ─── Phase 3: reverse KL over generated suffix only ───
        s_gen = s_logits[:, s_plen-1:-1, :].float()
        t_gen = t_logits[:, s_plen-1:-1, :].float()
        log_p_s = F.log_softmax(s_gen, dim=-1)
        log_p_t = F.log_softmax(t_gen, dim=-1)
        kl = F.kl_div(log_p_t, log_p_s, log_target=True, reduction="none").sum(-1)
        gen_mask = suffix.ne(tokenizer.pad_token_id).float()
        loss = (kl * gen_mask).sum() / gen_mask.sum().clamp(min=1.0)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  ⚠️  unstable step {step}, skipped")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
        opd_optim.step()
        opd_losses.append(loss.item())

        if step % 20 == 0:
            print(f"  OPD epoch {epoch+1} step {step:3d} | loss: {loss.item():.4f}")

print(f"\n✓ OPD done. Final avg (last 20): {np.mean(opd_losses[-20:]):.4f}")

SyntaxError: invalid syntax (808829589.py, line 4)

In [ ]:
# Final evaluation
del teacher_model
torch.cuda.empty_cache()

print("=" * 60)
print("POST-OPD (after recovery)")
print("=" * 60)

print("\n[IF eval] (expect: recovers toward baseline)")
post_opd_if = evaluate_if(student_model, verbose=True)
print(f"\n  → IF: {post_opd_if:.1f}%  (baseline {baseline_if:.1f} → post-SFT {post_sft_if:.1f} → now {post_opd_if:.1f})")

print("\n[JSON eval] (expect: mostly preserved)")
post_opd_json = evaluate_json(student_model, verbose=True)
print(f"\n  → JSON: {post_opd_json:.1f}%  (baseline {baseline_json:.1f} → post-SFT {post_sft_json:.1f} → now {post_opd_json:.1f})")

scores["post_opd"] = {"if": post_opd_if, "json": post_opd_json}

print("\n" + "=" * 60)
print("FINAL SCOREBOARD")
print("=" * 60)
print(f"{'Stage':12s} | {'IF (%)':>8s} | {'JSON (%)':>9s}")
print("-" * 36)
for s in ["baseline", "post_sft", "post_opd"]:
    print(f"{s:12s} | {scores[s]['if']:>8.1f} | {scores[s]['json']:>9.1f}")

In [ ]:
# Visualization
stages = ["baseline", "post_sft", "post_opd"]
labels = ["Baseline\n(original)", "Post-SFT\n(JSON trained)", "Post-OPD\n(recovered)"]
if_vals = [scores[s]["if"] for s in stages]
json_vals = [scores[s]["json"] for s in stages]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: behavior across stages
x = np.arange(len(stages)); w = 0.35
b1 = ax1.bar(x - w/2, if_vals,   w, label="IF (instruction following)", color="#1f77b4", edgecolor="black")
b2 = ax1.bar(x + w/2, json_vals, w, label="JSON (domain)",              color="#ff7f0e", edgecolor="black")
ax1.set_xticks(x); ax1.set_xticklabels(labels)
ax1.set_ylabel("Score (%)"); ax1.set_ylim(0, 105)
ax1.set_title("Catastrophic forgetting → recovery via self-distillation")
ax1.legend(); ax1.grid(axis="y", alpha=0.3)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 1.5, f"{h:.0f}", ha="center", fontsize=9)

# Loss curves: SFT then OPD
ax2.plot(sft_losses, label="SFT phase", color="#d62728", linewidth=1.0, alpha=0.85)
offset = len(sft_losses)
ax2.plot(np.arange(offset, offset + len(opd_losses)), opd_losses,
         label="OPD phase (recovery)", color="#2ca02c", linewidth=1.0, alpha=0.85)
ax2.axvline(offset, linestyle="--", color="gray", alpha=0.5)
ax2.set_xlabel("Training step (concatenated)"); ax2.set_ylabel("Loss")
ax2.set_yscale("log"); ax2.set_title("Training loss across phases (log scale)")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("continual_learning_results.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Save the recovered model
SAVE_DIR = "./qwen2.5-0.5b-json-continual-v1"
os.makedirs(SAVE_DIR, exist_ok=True)
student_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Model card
card = f"""---
base_model: Qwen/Qwen2.5-0.5B-Instruct
license: apache-2.0
tags: [continual-learning, on-policy-distillation, self-distillation]
---

# qwen2.5-0.5b-json-continual-v1

Reproduction of Thinking Machines' continual-learning experiment
("On-Policy Distillation", Lu et al., Oct 2025) at consumer scale.

## Method
1. SFT on synthetic JSON-emission task (800 examples, 1 epoch) →
   catastrophic forgetting of instruction-following.
2. On-policy reverse-KL distillation against the original Qwen2.5-0.5B-Instruct
   as teacher, on general instruction prompts (80 prompts, 2 epochs) →
   instruction-following recovered.

## Results
| Stage      | IF score | JSON score |
|------------|----------|------------|
| Baseline   | {baseline_if:.1f}%  | {baseline_json:.1f}%   |
| Post-SFT   | {post_sft_if:.1f}%  | {post_sft_json:.1f}%   |
| Post-OPD   | {post_opd_if:.1f}%  | {post_opd_json:.1f}%   |

## Hardware
NVIDIA GTX 1650 (4 GB), Windows + PowerShell.
Student in FP16, teacher in 4-bit NF4, optimizer = bitsandbytes PagedAdamW8bit.
"""
with open(os.path.join(SAVE_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(card)

print(f"✓ Saved to {SAVE_DIR}/")